# Alphalens Tearsheet Viewer

Load `factor_data.parquet` from `research/output/<run_tag>/`, render Alphalens tear sheets inline, and optionally save PNG files.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = None
for c in ROOT_CANDIDATES:
    if (c / 'research').is_dir() and (c / 'notebooks').is_dir():
        REPO_ROOT = c
        break
assert REPO_ROOT is not None, 'Run this notebook from repo root or notebooks/.'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUTPUT_ROOT = REPO_ROOT / 'research' / 'output'
assert OUTPUT_ROOT.exists(), f'Missing output directory: {OUTPUT_ROOT}'

print('repo_root:', REPO_ROOT)
print('output_root:', OUTPUT_ROOT)

In [ ]:
def discover_factor_runs(output_root: Path) -> list[Path]:
    runs = []
    for p in sorted(output_root.iterdir()):
        if p.is_dir() and (p / 'factor_data.parquet').exists():
            runs.append(p)
    return runs

RUN_DIRS = discover_factor_runs(OUTPUT_ROOT)
assert RUN_DIRS, 'No run directories with factor_data.parquet were found.'

print('Available runs (latest 10):')
for p in RUN_DIRS[-10:]:
    print(' -', p.name)

In [ ]:
# Set RUN_TAG for a specific run, else the latest discovered run is used.
RUN_TAG = None

if RUN_TAG:
    RUN_DIR = OUTPUT_ROOT / RUN_TAG
else:
    RUN_DIR = RUN_DIRS[-1]

FACTOR_DATA_PATH = RUN_DIR / 'factor_data.parquet'
assert FACTOR_DATA_PATH.exists(), f'Missing factor_data.parquet: {FACTOR_DATA_PATH}'

print('run_dir:', RUN_DIR)
print('factor_data:', FACTOR_DATA_PATH)

In [ ]:
factor_data = pd.read_parquet(FACTOR_DATA_PATH)
assert isinstance(factor_data.index, pd.MultiIndex) and factor_data.index.nlevels == 2, (
    'factor_data must use MultiIndex (date, asset)'
)

print('rows:', len(factor_data))
print('columns:', list(factor_data.columns))
factor_data.head()

In [ ]:
try:
    from alphalens import tears
except Exception as exc:
    raise RuntimeError(
        'alphalens-reloaded is required. Install with: ' \n        '.conda/tradingagents/bin/pip install alphalens-reloaded'
    ) from exc

LONG_SHORT = True
GROUP_NEUTRAL = False
BY_GROUP = False
TURNOVER_PERIODS = [5, 20]

tears.create_returns_tear_sheet(
    factor_data=factor_data,
    long_short=bool(LONG_SHORT),
    group_neutral=bool(GROUP_NEUTRAL),
    by_group=bool(BY_GROUP),
)
tears.create_information_tear_sheet(
    factor_data=factor_data,
    group_neutral=bool(GROUP_NEUTRAL),
    by_group=bool(BY_GROUP),
)
try:
    tears.create_turnover_tear_sheet(
        factor_data=factor_data,
        turnover_periods=list(TURNOVER_PERIODS),
    )
except Exception as exc:
    print('[warn] turnover tearsheet skipped:', exc)

print('rendered figures:', len(plt.get_fignums()))

In [ ]:
# Optional: save currently open figures as PNG files.
SAVE_PNG = False
PNG_DPI = 140
PNG_PREFIX = 'alphalens_nb'
PNG_DIR = RUN_DIR / 'tearsheet_png_notebook'

if SAVE_PNG:
    PNG_DIR.mkdir(parents=True, exist_ok=True)
    saved = []
    for i, fig_num in enumerate(plt.get_fignums(), start=1):
        fig = plt.figure(fig_num)
        path = PNG_DIR / f'{PNG_PREFIX}_{i:02d}.png'
        fig.savefig(path, dpi=int(PNG_DPI), bbox_inches='tight')
        saved.append(path)
    print(f'saved {len(saved)} figure(s) to {PNG_DIR}')
    for p in saved:
        print(' -', p)
else:
    print('Set SAVE_PNG=True to export PNG files.')